# Preparación de propiedades usadas para OLS

Entrada única: `PropUsadas_Integrado.csv`, sin limpieza previa. Se divide por tiempo antes de aprender parámetros; la limpieza determinística se aplica por igual a train y test. Se preparan matrices numéricas, sin Tukey, transformaciones del precio, segmentación, VIF, selección estadística ni entrenamiento de regresiones. El objetivo permanece en RUB.

Los parámetros se conservan en un registro funcional y un codificador dentro de un único Joblib. Las tablas Markdown se generan a partir de la ejecución y se guardan en celdas para facilitar su copia; deben actualizarse si cambia el CSV de entrada.

## 1. Carga y ordenamiento cronológico

Se convierte `date_posted` y se comprueba su concordancia con `date_posted_dt` si ya existe. Fechas inválidas, IDs ausentes/repetidos o duplicados exactos detienen el flujo para su revisión explícita; no se pierden filas silenciosamente. Se ordena por fecha e ID como desempate reproducible.

In [1]:
from pathlib import Path
import json
import hashlib
import warnings
import joblib
import numpy as np
import pandas as pd
from pandas.api.types import is_numeric_dtype, is_bool_dtype
from scipy.linalg import qr
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from importlib.metadata import version

raiz = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / "Ventas/Fase1_PropUsadas/data/PropUsadas_Integrado.csv").is_file()), None)
if raiz is None:
    raise FileNotFoundError("Ejecutar dentro del proyecto Moscow_EDA.")
entrada = raiz / "Ventas/Fase1_PropUsadas/data/PropUsadas_Integrado.csv"
carpeta_salida = raiz / "Ventas/Fase2_PropUsadas/data"
df_original = pd.read_csv(entrada)
tipos_originales = df_original.dtypes.astype(str).rename_axis("Columna").reset_index(name="Tipo")
if df_original.duplicated().any() or df_original.id.isna().any() or not df_original.id.is_unique:
    raise ValueError("Duplicados o IDs inválidos: revisar explícitamente antes de dividir.")
df_ordenado = df_original.copy(deep=True)
fechas = pd.to_datetime(df_ordenado.date_posted, errors="coerce")
if fechas.isna().any():
    raise ValueError(f"Fechas nulas o inválidas en date_posted: {fechas.isna().sum()}")
if "date_posted_dt" in df_ordenado:
    anteriores = pd.to_datetime(df_ordenado.date_posted_dt, errors="coerce")
    if anteriores.isna().any() or not anteriores.equals(fechas):
        raise ValueError("date_posted_dt no coincide con la fecha original: revisar antes de dividir.")
df_ordenado["date_posted_dt"] = fechas
df_ordenado = df_ordenado.sort_values(["date_posted_dt", "id"], kind="stable")
dimensiones_entrada = pd.DataFrame([{"Filas": len(df_original), "Columnas": df_original.shape[1],
                                    "Fechas inválidas": int(fechas.isna().sum())}])

def numero_md(valor, decimales=4):
    if valor is None or (not isinstance(valor, (str, list, dict)) and pd.isna(valor)):
        return "No aplica"
    if isinstance(valor, (bool, np.bool_)): return "Sí" if valor else "No"
    if isinstance(valor, (int, np.integer)): return f"{valor:,}".replace(",", ".")
    if isinstance(valor, (float, np.floating)):
        return f"{valor:,.{decimales}f}".translate(str.maketrans({",": ".", ".": ","}))
    return str(valor).replace("|", r"\|").replace("\n", " ")

def tabla_md(tabla):
    lineas = ["| " + " | ".join(map(str, tabla.columns)) + " |",
              "| " + " | ".join(["---"] * len(tabla.columns)) + " |"]
    for fila in tabla.itertuples(index=False, name=None):
        lineas.append("| " + " | ".join(numero_md(v) for v in fila) + " |")
    return "\n".join(lineas)

def a_json(obj):
    if isinstance(obj, dict): return {str(k): a_json(v) for k, v in obj.items()}
    if isinstance(obj, (list, tuple)): return [a_json(v) for v in obj]
    if isinstance(obj, np.generic): return obj.item()
    if isinstance(obj, Path): return str(obj)
    return obj

**Dimensiones de entrada**

| Filas | Columnas | Fechas inválidas |
| --- | --- | --- |
| 50.000 | 40 | 0 |

**Esquema original**

| Columna | Tipo |
| --- | --- |
| id | str |
| date_posted | str |
| district | str |
| okrug | str |
| lat | float64 |
| lon | float64 |
| total_area | float64 |
| living_area | float64 |
| kitchen_area | float64 |
| rooms | int64 |
| floor | int64 |
| total_floors | int64 |
| building_year | int64 |
| building_type | str |
| ceiling_height | float64 |
| has_balcony | bool |
| renovation | str |
| metro_station | str |
| metro_line | str |
| metro_distance_min | int64 |
| metro_distance_type | str |
| to_center_km | float64 |
| price_rub | int64 |
| price_per_sqm | int64 |
| mortgage_rate_at_listing | float64 |
| seller_type | str |
| date_posted_dt | str |
| year_month | str |
| secondary_price_per_sqm | int64 |
| newbuild_price_per_sqm | int64 |
| secondary_mom_change_pct | float64 |
| newbuild_mom_change_pct | float64 |
| n_listings_secondary | int64 |
| n_listings_newbuild | int64 |
| cbr_key_rate_pct | float64 |
| avg_mortgage_rate_pct | float64 |
| metro_lat | float64 |
| metro_lon | float64 |
| metro_year_opened | int64 |
| metro_to_center_km | float64 |

## 2. División temporal 80/20

El primer 80 % cronológico es train y el resto es test. En este archivo son 40.000 y 10.000 filas. Si el corte atraviesa un día, esa fecha puede figurar en ambos rangos; los IDs son disjuntos. A partir de aquí no se estiman parámetros con test.

In [2]:
corte = int(len(df_ordenado) * 0.80)
train_crudo = df_ordenado.iloc[:corte].copy(deep=True)
test_crudo = df_ordenado.iloc[corte:].copy(deep=True)
assert len(train_crudo) == 40_000 and len(test_crudo) == 10_000
assert set(train_crudo.id).isdisjoint(test_crudo.id)
assert train_crudo.date_posted_dt.max() <= test_crudo.date_posted_dt.min()
resumen_particion = pd.DataFrame([
    {"Conjunto": nombre, "Cantidad": len(datos), "Fecha mínima": datos.date_posted_dt.min().strftime("%Y-%m-%d"),
     "Fecha máxima": datos.date_posted_dt.max().strftime("%Y-%m-%d")}
    for nombre, datos in [("Train", train_crudo), ("Test", test_crudo)]])

**Partición temporal**

| Conjunto | Cantidad | Fecha mínima | Fecha máxima |
| --- | --- | --- | --- |
| Train | 40.000 | 2020-01-01 | 2025-01-17 |
| Test | 10.000 | 2025-01-17 | 2026-04-29 |

## 3. Limpieza y validación

Se reproducen las reglas de `Analisis_Usadas`: conversión del esquema numérico, infinitos a nulos, corrección exclusiva de `total_area` si las tres superficies son positivas y finitas y el total es menor que su suma. Pisos y años deben ser enteros: pisos positivos y `floor ≤ total_floors` cuando el total es válido; año de cuatro cifras (≥1000) no posterior al anuncio. El año mínimo es un control de formato, no un supuesto histórico.

Los inválidos se convierten en nulos, conservando las filas. Si el piso supera un total válido se anula el piso. Precios faltantes, no finitos o no positivos se registran y detienen la preparación: el objetivo no se imputa ni se transforma.

In [3]:
columnas_numericas = [
    "lat", "lon", "total_area", "living_area", "kitchen_area", "rooms",
    "floor", "total_floors", "building_year", "ceiling_height",
    "metro_distance_min", "to_center_km", "price_rub", "price_per_sqm",
    "mortgage_rate_at_listing", "secondary_price_per_sqm", "newbuild_price_per_sqm",
    "secondary_mom_change_pct", "newbuild_mom_change_pct", "n_listings_secondary",
    "n_listings_newbuild", "cbr_key_rate_pct", "avg_mortgage_rate_pct",
    "metro_lat", "metro_lon", "metro_year_opened", "metro_to_center_km"
]

def limpiar_validar(datos, esquema_numerico=None):
    salida = datos.copy(deep=True)
    auditoria = {"Filas": len(salida)}
    for col in (columnas_numericas if esquema_numerico is None else esquema_numerico):
        salida[col] = pd.to_numeric(salida[col], errors="coerce").replace([np.inf, -np.inf], np.nan)
    invalido_objetivo = salida.price_rub.isna() | salida.price_rub.le(0)
    auditoria["Precios inválidos"] = int(invalido_objetivo.sum())
    if invalido_objetivo.any():
        raise ValueError(f"Objetivos inválidos, sin imputar: {salida.loc[invalido_objetivo, 'id'].tolist()[:10]}")
    areas = salida[["total_area", "living_area", "kitchen_area"]]
    validas = (np.isfinite(areas) & areas.gt(0)).all(axis=1)
    corregir = validas & salida.total_area.lt(salida.living_area + salida.kitchen_area)
    salida.loc[corregir, "total_area"] = salida.loc[corregir, "living_area"] + salida.loc[corregir, "kitchen_area"]
    auditoria["Superficies corregidas"] = int(corregir.sum())
    t, f, y = salida.total_floors, salida.floor, salida.building_year
    total_valido = np.isfinite(t) & t.gt(0) & t.mod(1).eq(0)
    piso_valido = np.isfinite(f) & f.gt(0) & f.mod(1).eq(0)
    mascaras = {"total_floors": t.notna() & ~total_valido,
        "floor": f.notna() & (~piso_valido | (total_valido & f.gt(t))),
        "building_year": y.notna() & (~np.isfinite(y) | y.mod(1).ne(0) | y.lt(1000) | y.gt(salida.date_posted_dt.dt.year))}
    for col, mascara in mascaras.items():
        auditoria[f"{col}: convertidos a nulo"] = int(mascara.sum())
        salida.loc[mascara, col] = np.nan
    return salida, auditoria

train_validado, limpieza_train = limpiar_validar(train_crudo)
test_validado, limpieza_test = limpiar_validar(test_crudo)
resumen_limpieza = pd.DataFrame([{"Conjunto": "Train", **limpieza_train}, {"Conjunto": "Test", **limpieza_test}])

**Correcciones determinísticas por conjunto**

| Conjunto | Filas | Precios inválidos | Superficies corregidas | total_floors: convertidos a nulo | floor: convertidos a nulo | building_year: convertidos a nulo |
| --- | --- | --- | --- | --- | --- | --- |
| Train | 40.000 | 0 | 186 | 0 | 0 | 1.271 |
| Test | 10.000 | 0 | 61 | 0 | 0 | 0 |

## 4. Imputación de valores faltantes

Se aprende una sola vez sobre train: mediana numérica, media para `secondary_mom_change_pct`, moda para binarias y `__FALTANTE__` para nominales. Son binarias las booleanas, las codificadas 0/1 y las textuales con dos categorías (excepto `district`, que conserva tratamiento nominal independiente). Las fechas, el ID y el objetivo se excluyen de la imputación.

Una variable numérica o binaria totalmente vacía en train queda registrada como no estimable y detiene el flujo para revisión explícita; no se toma un valor de test. Las verificaciones posteriores también detienen el flujo si una imputación viola las relaciones entre pisos o años, sin inventar ajustes adicionales. En este archivo no se presentan esos casos.

In [4]:
def es_binaria_train(col, serie):
    valores = serie.dropna()
    if col == "district": return False
    if is_bool_dtype(serie) or col == "has_balcony": return True
    if is_numeric_dtype(serie): return len(valores) > 0 and valores.isin([0, 1]).all()
    return valores.nunique() == 2

excluir_imputacion = {"id", "date_posted", "date_posted_dt", "price_rub"}
valores_imputacion = {}
roles_imputacion = {}
sin_parametro_train = []
for col in train_validado:
    if col in excluir_imputacion: continue
    serie = train_validado[col]
    if es_binaria_train(col, serie):
        tipo, estrategia = "Binaria", "Moda de train"
        modas = serie.mode(dropna=True)
        valor = modas.iloc[0] if len(modas) else np.nan
    elif is_numeric_dtype(serie):
        tipo = "Numérica"
        estrategia = "Media de train" if col == "secondary_mom_change_pct" else "Mediana de train"
        valor = serie.mean() if col == "secondary_mom_change_pct" else serie.median()
    else:
        tipo, estrategia, valor = "Nominal", "Categoría fija", "__FALTANTE__"
    roles_imputacion[col] = {"tipo": tipo, "estrategia": estrategia}
    if pd.isna(valor): sin_parametro_train.append(col)
    valores_imputacion[col] = valor
if sin_parametro_train:
    raise ValueError(f"Variables sin parámetro estimable en train: {sin_parametro_train}; revisar antes de continuar.")

def aplicar_imputacion(datos, parametros):
    salida = datos.copy(deep=True)
    for col, valor in parametros.items():
        salida[col] = salida[col].fillna(valor)
    return salida

train_imputado = aplicar_imputacion(train_validado, valores_imputacion)
test_imputado = aplicar_imputacion(test_validado, valores_imputacion)
filas_imputacion = []
for col, valor in valores_imputacion.items():
    filas_imputacion.append({"Variable": col, **roles_imputacion[col], "Valor aprendido": valor,
        "Nulos train": int(train_validado[col].isna().sum()), "Nulos test": int(test_validado[col].isna().sum()),
        "Restantes train": int(train_imputado[col].isna().sum()), "Restantes test": int(test_imputado[col].isna().sum())})
registro_imputacion = pd.DataFrame(filas_imputacion)
tabla_imputacion = registro_imputacion.loc[(registro_imputacion["Nulos train"] + registro_imputacion["Nulos test"]).gt(0)]

def comprobar_consistencia(datos):
    assert not datos.isna().any().any()
    assert (datos.floor.gt(0) & datos.total_floors.gt(0) & datos.floor.le(datos.total_floors)).all()
    assert datos[["floor", "total_floors", "building_year"]].mod(1).eq(0).all().all()
    assert (datos.building_year.ge(1000) & datos.building_year.le(datos.date_posted_dt.dt.year)).all()
    assert np.isfinite(datos.select_dtypes(include="number")).all().all()

comprobar_consistencia(train_imputado)
comprobar_consistencia(test_imputado)
pd.testing.assert_series_equal(train_imputado.price_rub, train_crudo.price_rub)
pd.testing.assert_series_equal(test_imputado.price_rub, test_crudo.price_rub)

**Variables que requirieron imputación**

| Variable | tipo | estrategia | Valor aprendido | Nulos train | Nulos test | Restantes train | Restantes test |
| --- | --- | --- | --- | --- | --- | --- | --- |
| building_year | Numérica | Mediana de train | 1.988,0000 | 1.271 | 0 | 0 | 0 |
| secondary_mom_change_pct | Numérica | Media de train | 0,7333 | 678 | 0 | 0 | 0 |
| newbuild_mom_change_pct | Numérica | Mediana de train | 0,5900 | 678 | 0 | 0 | 0 |

## 5. Eliminación de variables

La superficie habitable se utiliza antes de eliminarla. Se conserva `district`; `mortgage_rate_at_listing` pasa a llamarse `mortgage_key_rate_at_listing`, sin cambiar sus valores. Se conservan `total_area`, `to_center_km`, la fecha y el ID; estos dos últimos se separarán de los predictores. El objetivo siempre queda aparte.

In [5]:
import re

motivos_eliminacion = {
    "rooms": "Se prioriza superficie; evitar redundancia entre características relacionadas.",
    "living_area": "Utilizada para corregir total_area; se conserva la superficie total.",
    "avg_mortgage_rate_pct": "Tasa hipotecaria redundante; se prioriza mortgage_key_rate_at_listing.",
    "cbr_key_rate_pct": "Tasa del Banco Central; se prioriza mortgage_key_rate_at_listing.",
    "date_posted": "La referencia temporal es date_posted_dt.",
    "okrug": "Fuera de los predictores; district se codifica de forma independiente.",
    "lat": "Se priorizan otras medidas de ubicación y accesibilidad.",
    "lon": "Se priorizan otras medidas de ubicación y accesibilidad.",
    "metro_station": "Alta cardinalidad; información parcialmente representada por accesibilidad y distancia.",
    "price_per_sqm": "Derivada del objetivo; riesgo de fuga de información.",
    "year_month": "Se generan año y mes desde date_posted_dt.",
    "secondary_price_per_sqm": "No se utilizan precios por unidad de superficie.",
    "newbuild_price_per_sqm": "Precios de construcciones nuevas por unidad de superficie.",
    "newbuild_mom_change_pct": "Se conserva solo variación mensual del mercado secundario.",
    "n_listings_secondary": "Fuera de los predictores definidos para este flujo.",
    "n_listings_newbuild": "Cantidad de anuncios de construcciones nuevas fuera del enfoque.",
    "metro_lat": "Se priorizan otras medidas de ubicación y accesibilidad.",
    "metro_lon": "Se priorizan otras medidas de ubicación y accesibilidad.",
    "metro_year_opened": "Año de apertura fuera del enfoque actual."}
for col in train_imputado:
    if "newbuild" in re.sub(r"[^a-z0-9]", "", col.lower()):
        motivos_eliminacion.setdefault(col, "Variable del segmento de construcciones nuevas.")
columnas_eliminadas = [c for c in train_imputado if c in motivos_eliminacion]
tabla_eliminadas = pd.DataFrame([{"Variable eliminada": c, "Justificación": motivos_eliminacion[c]} for c in columnas_eliminadas])
alias_variables = {"mortgage_rate_at_listing": "mortgage_key_rate_at_listing"}
def eliminar_y_renombrar(datos):
    return datos.drop(columns=columnas_eliminadas, errors="ignore").rename(columns=alias_variables).copy()

train_preparado = eliminar_y_renombrar(train_imputado)
test_preparado = eliminar_y_renombrar(test_imputado)
assert not set(columnas_eliminadas).intersection(train_preparado)
assert {"district", "mortgage_key_rate_at_listing", "to_center_km", "total_area", "price_rub", "date_posted_dt", "id"}.issubset(train_preparado)

**Variables eliminadas y justificación**

| Variable eliminada | Justificación |
| --- | --- |
| date_posted | La referencia temporal es date_posted_dt. |
| okrug | Fuera de los predictores; district se codifica de forma independiente. |
| lat | Se priorizan otras medidas de ubicación y accesibilidad. |
| lon | Se priorizan otras medidas de ubicación y accesibilidad. |
| living_area | Utilizada para corregir total_area; se conserva la superficie total. |
| rooms | Se prioriza superficie; evitar redundancia entre características relacionadas. |
| metro_station | Alta cardinalidad; información parcialmente representada por accesibilidad y distancia. |
| price_per_sqm | Derivada del objetivo; riesgo de fuga de información. |
| year_month | Se generan año y mes desde date_posted_dt. |
| secondary_price_per_sqm | No se utilizan precios por unidad de superficie. |
| newbuild_price_per_sqm | Precios de construcciones nuevas por unidad de superficie. |
| newbuild_mom_change_pct | Se conserva solo variación mensual del mercado secundario. |
| n_listings_secondary | Fuera de los predictores definidos para este flujo. |
| n_listings_newbuild | Cantidad de anuncios de construcciones nuevas fuera del enfoque. |
| cbr_key_rate_pct | Tasa del Banco Central; se prioriza mortgage_key_rate_at_listing. |
| avg_mortgage_rate_pct | Tasa hipotecaria redundante; se prioriza mortgage_key_rate_at_listing. |
| metro_lat | Se priorizan otras medidas de ubicación y accesibilidad. |
| metro_lon | Se priorizan otras medidas de ubicación y accesibilidad. |
| metro_year_opened | Año de apertura fuera del enfoque actual. |

## 6. Transformación de características

Se generan año y mes de publicación. Las nominales restantes usan One-Hot con una referencia excluida por variable; `district` tiene su propio bloque. Las categorías se aprenden solo de train. Una categoría desconocida en test se representa con ceros en su bloque (misma representación que la referencia) y queda registrada, sin nuevas columnas.

Las binarias usan una sola columna 0/1: se preserva la codificación numérica existente o se asigna 0/1 por orden de categorías de train. Una categoría binaria desconocida utiliza el código 0 de referencia y se registra. No se impone orden a tipo de edificio, línea de metro ni renovación: “designer” y “euro” describen categorías que no justifican distancias ordinales. No hay ordinales en este flujo.

`Pipeline` y `ColumnTransformer` se ajustan **una sola vez** con train; test solo recibe `transform`. Se excluyen columnas constantes aprendidas en train. El rango con intercepto se comprueba para detectar dependencias exactas, sin VIF ni selección estadística. Una dependencia no resuelta detiene el flujo para revisión. No se normalizan ni estandarizan las matrices exportadas.

In [6]:
variables_temporales = {"anio_publicacion": "date_posted_dt.dt.year", "mes_publicacion": "date_posted_dt.dt.month"}
def generar_temporales(datos):
    salida = datos.drop(columns=["id", "price_rub", "date_posted_dt"]).copy()
    salida["anio_publicacion"] = datos.date_posted_dt.dt.year
    salida["mes_publicacion"] = datos.date_posted_dt.dt.month
    return salida

train_entrada = generar_temporales(train_preparado)
test_entrada = generar_temporales(test_preparado)
binarias = [c for c in train_entrada if es_binaria_train(c, train_entrada[c])]
nominales = [c for c in train_entrada if c not in binarias and not is_numeric_dtype(train_entrada[c])]
numericas = [c for c in train_entrada if c not in binarias + nominales]
ordinales = []
mapas_binarios = {}
for col in binarias:
    valores = sorted(train_entrada[col].dropna().unique().tolist())
    if is_numeric_dtype(train_entrada[col]) or is_bool_dtype(train_entrada[col]):
        mapas_binarios[col] = [{"categoria": v, "valor": int(v)} for v in valores]
    else:
        mapas_binarios[col] = [{"categoria": v, "valor": i} for i, v in enumerate(valores)]

def aplicar_binarias(datos, mapas=None):
    salida = datos.copy(deep=True)
    desconocidas = {}
    for col, pares in (mapas_binarios if mapas is None else mapas).items():
        mapa = {p["categoria"]: p["valor"] for p in pares}
        desconocidas[col] = int((~salida[col].isin(mapa)).sum())
        salida[col] = salida[col].map(mapa).fillna(0).astype(float)
    return salida, desconocidas

train_codificacion, _ = aplicar_binarias(train_entrada)
test_codificacion, binarias_desconocidas_test = aplicar_binarias(test_entrada)
bloques = [("numericas", "passthrough", numericas), ("binarias", "passthrough", binarias)]
nombres_bloque_nominal = {}
for i, col in enumerate(nominales):
    nombre = "district" if col == "district" else f"nominal_{i}"
    nombres_bloque_nominal[col] = nombre
    bloques.append((nombre, OneHotEncoder(drop="first", handle_unknown="ignore", sparse_output=False, dtype=float), [col]))
codificador = Pipeline([("columnas", ColumnTransformer(bloques, remainder="drop", verbose_feature_names_out=False))])
matriz_train = codificador.fit_transform(train_codificacion)
with warnings.catch_warnings():
    warnings.filterwarnings("ignore", message="Found unknown categories", category=UserWarning)
    matriz_test = codificador.transform(test_codificacion)
columnas_generadas = codificador.get_feature_names_out().tolist()
X_train_completo = pd.DataFrame(matriz_train, index=train_preparado.index, columns=columnas_generadas, dtype=float)
X_test_completo = pd.DataFrame(matriz_test, index=test_preparado.index, columns=columnas_generadas, dtype=float)
columnas_constantes = X_train_completo.columns[X_train_completo.nunique(dropna=False).le(1)].tolist()
columnas_finales = [c for c in columnas_generadas if c not in columnas_constantes]
X_train = X_train_completo[columnas_finales].copy()
X_test = X_test_completo[columnas_finales].copy()
y_train = train_preparado.price_rub.copy()
y_test = test_preparado.price_rub.copy()
meta_train = train_preparado[["id", "date_posted_dt"]].copy()
meta_test = test_preparado[["id", "date_posted_dt"]].copy()

categorias_nominales, referencias_nominales, dummies_por_variable = {}, {}, {}
desconocidas_test = []
for col, nombre in nombres_bloque_nominal.items():
    encoder = codificador.named_steps["columnas"].named_transformers_[nombre]
    categorias_nominales[col] = encoder.categories_[0].tolist()
    referencias_nominales[col] = encoder.categories_[0][encoder.drop_idx_[0]]
    dummies_por_variable[col] = encoder.get_feature_names_out([col]).tolist()
    desconocidas_test.append({"Variable": col, "Desconocidas test": int((~test_entrada[col].isin(categorias_nominales[col])).sum())})
tabla_categorias = pd.DataFrame([{"Variable": c, "Categorías en train": len(categorias_nominales[c]),
    "Referencia excluida": referencias_nominales[c], "Dummies": len(dummies_por_variable[c])} for c in nominales])
tabla_binarias = pd.DataFrame([{"Variable": c, "Correspondencia aprendida": "; ".join(
    f"{p['categoria']} → {p['valor']}" for p in mapas_binarios[c])} for c in binarias])
tabla_desconocidas = pd.DataFrame(desconocidas_test + [{"Variable": c, "Desconocidas test": n} for c, n in binarias_desconocidas_test.items()])

# Solo comprobación numérica de rango: los datos de X no se escalan ni se modifican.
comprobacion = np.column_stack([np.ones(len(X_train)), X_train.to_numpy()])
normas = np.linalg.norm(comprobacion, axis=0)
_, triangular, pivote = qr(comprobacion / normas, mode="economic", pivoting=True, check_finite=False)
tolerancia_rango = np.finfo(float).eps * max(comprobacion.shape) * np.abs(np.diag(triangular)).max()
rango_con_intercepto = int((np.abs(np.diag(triangular)) > tolerancia_rango).sum())
dependencias_exactas = []
if rango_con_intercepto != comprobacion.shape[1]:
    dependencias_exactas = [["intercepto", *columnas_finales][i] for i in pivote[rango_con_intercepto:]]
    raise ValueError(f"Dependencias lineales exactas pendientes de revisión: {dependencias_exactas}")
del comprobacion, normas, triangular, matriz_train, matriz_test
tabla_estructura = pd.DataFrame([{"Constantes excluidas": len(columnas_constantes),
    "Dependencias exactas pendientes": len(dependencias_exactas), "Rango con intercepto": rango_con_intercepto}])

**Categorías nominales y referencias de train**

| Variable | Categorías en train | Referencia excluida | Dummies |
| --- | --- | --- | --- |
| district | 129 | Aeroport | 128 |
| building_type | 6 | brick | 5 |
| renovation | 4 | cosmetic | 3 |
| metro_line | 12 | Arbatsko-Pokrovskaya | 11 |

**Codificaciones binarias**

| Variable | Correspondencia aprendida |
| --- | --- |
| has_balcony | False → 0; True → 1 |
| metro_distance_type | transport → 0; walk → 1 |
| seller_type | agency → 0; owner → 1 |

**Categorías desconocidas al transformar test**

| Variable | Desconocidas test |
| --- | --- |
| district | 0 |
| building_type | 0 |
| renovation | 0 |
| metro_line | 0 |
| has_balcony | 0 |
| metro_distance_type | 0 |
| seller_type | 0 |

**Controles de estructura**

| Constantes excluidas | Dependencias exactas pendientes | Rango con intercepto |
| --- | --- | --- |
| 0 | 0 | 164 |

## 7. Registro de transformaciones

`registro_transformaciones` documenta el esquema, reglas, parámetros, alias, categorías, referencias, dummies, codificaciones, variables temporales y orden final. El artefacto `transformadores_ols_usadas.joblib` conserva el codificador y únicamente el registro necesario para aplicar las transformaciones. Las métricas y auditorías permanecen en las tablas del notebook. Las funciones determinísticas y de imputación de este notebook aplican los parámetros guardados, sin nuevos ajustes.

In [7]:
registro_transformaciones = a_json({
    "version_flujo": 1, "entrada": "Ventas/Fase1_PropUsadas/data/PropUsadas_Integrado.csv",
    "sha256_entrada": hashlib.sha256(entrada.read_bytes()).hexdigest(),
    "versiones": {p: version(p) for p in ["numpy", "pandas", "scipy", "scikit-learn", "joblib"]},
    "columnas_originales": df_original.columns.tolist(), "tipos_originales": df_original.dtypes.astype(str).to_dict(),
    "columnas_numericas_entrada": columnas_numericas,
    "particion": {"criterio": "date_posted_dt ascendente, id como desempate", "fraccion_train": 0.8,
                  "train": len(train_crudo), "test": len(test_crudo), "rango_fechas": resumen_particion.to_dict("records")},
    "correcciones": {"total_area": "max(total_area, living_area + kitchen_area) solo si las tres áreas son positivas y finitas",
        "pisos": "enteros positivos; anular floor si supera total_floors válido",
        "building_year": "entero >=1000 y <= año de publicación",
        "no_finitos": "convertir a nulo", "objetivo_invalido": "registrar y detener; no imputar"},
    "auditoria_limpieza": resumen_limpieza.to_dict("records"),
    "imputacion": {c: {**roles_imputacion[c], "valor": v} for c, v in valores_imputacion.items()},
    "auditoria_imputacion": registro_imputacion.to_dict("records"), "sin_parametro_train": sin_parametro_train,
    "eliminadas": columnas_eliminadas, "justificaciones": {c: motivos_eliminacion[c] for c in columnas_eliminadas},
    "alias": alias_variables, "nominales": categorias_nominales, "referencias": referencias_nominales,
    "binarias": mapas_binarios, "ordinales": {},
    "desconocidas": {"nominales": "ceros en bloque, equivalente a referencia", "binarias": "código 0, registrado"},
    "desconocidas_test": tabla_desconocidas.to_dict("records"),
    "dummies": dummies_por_variable, "temporales": variables_temporales,
    "entrada_codificador": train_codificacion.columns.tolist(),
    "columnas_generadas": columnas_generadas, "constantes_excluidas": columnas_constantes,
    "dependencias_exactas": dependencias_exactas, "rango_con_intercepto": rango_con_intercepto,
    "columnas_finales": columnas_finales, "objetivo": "price_rub", "metadatos": ["id", "date_posted_dt"],
    "ajuste": "Un único fit sobre train; test solo transform", "precio_transformado": False})

## 8. Resumen cuantitativo de predictores y dummies

Cada fila representa una característica original que participa de la preparación predictora; `date_posted_dt` genera dos columnas temporales. `id` y `price_rub` no son predictores. Las columnas finales reflejan cualquier constante excluida. Se muestran por separado todas las dummies de `district`, sin eliminar distritos por frecuencia.

In [8]:
filas_caracteristicas = []
for col in train_preparado:
    if col in ["id", "price_rub"]: continue
    if col == "date_posted_dt":
        tipo, categorias, codificacion, dummies, finales = "Temporal", None, "Año y mes", 0, sum(c in columnas_finales for c in variables_temporales)
    elif col in nominales:
        tipo, categorias, codificacion = "Nominal", len(categorias_nominales[col]), "One-Hot (k−1)"
        dummies = len(dummies_por_variable[col])
        finales = sum(c in columnas_finales for c in dummies_por_variable[col])
    elif col in binarias:
        tipo, categorias, codificacion, dummies, finales = "Binaria", len(mapas_binarios[col]), "0/1", 0, int(col in columnas_finales)
    else:
        tipo, categorias, codificacion, dummies, finales = "Numérica", None, "Sin codificación", 0, int(col in columnas_finales)
    filas_caracteristicas.append({"Variable original": col, "Tipo": tipo, "Categorías en train": categorias,
        "Codificación": codificacion, "Dummies generadas": dummies, "Columnas finales": finales})
tabla_caracteristicas = pd.DataFrame(filas_caracteristicas)
tabla_caracteristicas["Categorías en train"] = tabla_caracteristicas["Categorías en train"].map(
    lambda x: "No aplica" if pd.isna(x) else str(int(x)))
tabla_resumen = pd.DataFrame([{"Indicador": k, "Cantidad": v} for k, v in {
    "Columnas iniciales": df_original.shape[1], "Variables eliminadas": len(columnas_eliminadas),
    "Características originales predictoras conservadas (sin ID, objetivo ni fecha)": len(train_preparado.columns) - 3,
    "Numéricas originales conservadas": len([c for c in numericas if c not in variables_temporales]),
    "Temporales derivadas": len(variables_temporales), "Numéricas incluyendo temporales": len(numericas),
    "Binarias": len(binarias), "Nominales": len(nominales), "Ordinales": len(ordinales),
    "Dummies generadas": sum(map(len, dummies_por_variable.values())),
    "Constantes excluidas": len(columnas_constantes), "Predictores finales para OLS": X_train.shape[1]}.items()])
assert tabla_caracteristicas["Columnas finales"].sum() == X_train.shape[1]
tabla_district = pd.DataFrame([{"Columna dummy": dummy, "Categoría": cat,
    "Referencia excluida": referencias_nominales["district"]}
    for dummy, cat in zip(dummies_por_variable["district"],
        [c for c in categorias_nominales["district"] if c != referencias_nominales["district"]])])
assert len(tabla_district) == len(categorias_nominales["district"]) - 1

**Características originales y columnas resultantes**

| Variable original | Tipo | Categorías en train | Codificación | Dummies generadas | Columnas finales |
| --- | --- | --- | --- | --- | --- |
| district | Nominal | 129 | One-Hot (k−1) | 128 | 128 |
| total_area | Numérica | No aplica | Sin codificación | 0 | 1 |
| kitchen_area | Numérica | No aplica | Sin codificación | 0 | 1 |
| floor | Numérica | No aplica | Sin codificación | 0 | 1 |
| total_floors | Numérica | No aplica | Sin codificación | 0 | 1 |
| building_year | Numérica | No aplica | Sin codificación | 0 | 1 |
| building_type | Nominal | 6 | One-Hot (k−1) | 5 | 5 |
| ceiling_height | Numérica | No aplica | Sin codificación | 0 | 1 |
| has_balcony | Binaria | 2 | 0/1 | 0 | 1 |
| renovation | Nominal | 4 | One-Hot (k−1) | 3 | 3 |
| metro_line | Nominal | 12 | One-Hot (k−1) | 11 | 11 |
| metro_distance_min | Numérica | No aplica | Sin codificación | 0 | 1 |
| metro_distance_type | Binaria | 2 | 0/1 | 0 | 1 |
| to_center_km | Numérica | No aplica | Sin codificación | 0 | 1 |
| mortgage_key_rate_at_listing | Numérica | No aplica | Sin codificación | 0 | 1 |
| seller_type | Binaria | 2 | 0/1 | 0 | 1 |
| date_posted_dt | Temporal | No aplica | Año y mes | 0 | 2 |
| secondary_mom_change_pct | Numérica | No aplica | Sin codificación | 0 | 1 |
| metro_to_center_km | Numérica | No aplica | Sin codificación | 0 | 1 |

**Resumen cuantitativo**

| Indicador | Cantidad |
| --- | --- |
| Columnas iniciales | 40 |
| Variables eliminadas | 19 |
| Características originales predictoras conservadas (sin ID, objetivo ni fecha) | 18 |
| Numéricas originales conservadas | 11 |
| Temporales derivadas | 2 |
| Numéricas incluyendo temporales | 13 |
| Binarias | 3 |
| Nominales | 4 |
| Ordinales | 0 |
| Dummies generadas | 147 |
| Constantes excluidas | 0 |
| Predictores finales para OLS | 163 |

**Dummies de district y referencia excluida**

| Columna dummy | Categoría | Referencia excluida |
| --- | --- | --- |
| district_Akademichesky | Akademichesky | Aeroport |
| district_Alekseevsky | Alekseevsky | Aeroport |
| district_Altufyevsky | Altufyevsky | Aeroport |
| district_Arbat | Arbat | Aeroport |
| district_Babushkinsky | Babushkinsky | Aeroport |
| district_Basmanny | Basmanny | Aeroport |
| district_Begovoy | Begovoy | Aeroport |
| district_Beskudnikovsky | Beskudnikovsky | Aeroport |
| district_Bibirevo | Bibirevo | Aeroport |
| district_Biryulyovo Vostochnoye | Biryulyovo Vostochnoye | Aeroport |
| district_Biryulyovo Zapadnoye | Biryulyovo Zapadnoye | Aeroport |
| district_Bogorodskoye | Bogorodskoye | Aeroport |
| district_Brateyevo | Brateyevo | Aeroport |
| district_Butyrsky | Butyrsky | Aeroport |
| district_Chertanovo Severnoye | Chertanovo Severnoye | Aeroport |
| district_Chertanovo Tsentralnoye | Chertanovo Tsentralnoye | Aeroport |
| district_Chertanovo Yuzhnoye | Chertanovo Yuzhnoye | Aeroport |
| district_Cheryomushki | Cheryomushki | Aeroport |
| district_Danilovsky | Danilovsky | Aeroport |
| district_Desyonovskoye | Desyonovskoye | Aeroport |
| district_Dmitrovsky | Dmitrovsky | Aeroport |
| district_Donskoy | Donskoy | Aeroport |
| district_Dorogomilovo | Dorogomilovo | Aeroport |
| district_Fili-Davydkovo | Fili-Davydkovo | Aeroport |
| district_Filimonkovskoye | Filimonkovskoye | Aeroport |
| district_Filyovsky Park | Filyovsky Park | Aeroport |
| district_Gagarinsky | Gagarinsky | Aeroport |
| district_Golovinsky | Golovinsky | Aeroport |
| district_Golyanovo | Golyanovo | Aeroport |
| district_Ivanovskoye | Ivanovskoye | Aeroport |
| district_Izmaylovo | Izmaylovo | Aeroport |
| district_Khamovniki | Khamovniki | Aeroport |
| district_Khoroshyovo-Mnyovniki | Khoroshyovo-Mnyovniki | Aeroport |
| district_Khoroshyovsky | Khoroshyovsky | Aeroport |
| district_Khovrino | Khovrino | Aeroport |
| district_Kievsky | Kievsky | Aeroport |
| district_Klyonovskoye | Klyonovskoye | Aeroport |
| district_Kokoshkino | Kokoshkino | Aeroport |
| district_Konkovo | Konkovo | Aeroport |
| district_Koptevo | Koptevo | Aeroport |
| district_Kosino-Ukhtomsky | Kosino-Ukhtomsky | Aeroport |
| district_Kotlovka | Kotlovka | Aeroport |
| district_Krasnopakhorskoye | Krasnopakhorskoye | Aeroport |
| district_Krasnoselsky | Krasnoselsky | Aeroport |
| district_Krylatskoye | Krylatskoye | Aeroport |
| district_Kuntsevo | Kuntsevo | Aeroport |
| district_Kurkino | Kurkino | Aeroport |
| district_Levoberezhny | Levoberezhny | Aeroport |
| district_Lianozovo | Lianozovo | Aeroport |
| district_Lomonosovsky | Lomonosovsky | Aeroport |
| district_Losinoostrovsky | Losinoostrovsky | Aeroport |
| district_Marfino | Marfino | Aeroport |
| district_Marushkinskoye | Marushkinskoye | Aeroport |
| district_Maryina Roshcha | Maryina Roshcha | Aeroport |
| district_Meshchansky | Meshchansky | Aeroport |
| district_Metrogorodok | Metrogorodok | Aeroport |
| district_Mikhaylovo-Yartsevskoye | Mikhaylovo-Yartsevskoye | Aeroport |
| district_Mitino | Mitino | Aeroport |
| district_Molzhaninovsky | Molzhaninovsky | Aeroport |
| district_Moskovsky | Moskovsky | Aeroport |
| district_Moskvorechye-Saburovo | Moskvorechye-Saburovo | Aeroport |
| district_Mosrentgen | Mosrentgen | Aeroport |
| district_Mozhaysky | Mozhaysky | Aeroport |
| district_Nagatino-Sadovniki | Nagatino-Sadovniki | Aeroport |
| district_Nagatinsky Zaton | Nagatinsky Zaton | Aeroport |
| district_Nagorny | Nagorny | Aeroport |
| district_Novo-Peredelkino | Novo-Peredelkino | Aeroport |
| district_Novofyodorovskoye | Novofyodorovskoye | Aeroport |
| district_Novogireyevo | Novogireyevo | Aeroport |
| district_Novokosino | Novokosino | Aeroport |
| district_Obruchevsky | Obruchevsky | Aeroport |
| district_Ochakovo-Matveyevskoye | Ochakovo-Matveyevskoye | Aeroport |
| district_Orekhovo-Borisovo Severnoye | Orekhovo-Borisovo Severnoye | Aeroport |
| district_Orekhovo-Borisovo Yuzhnoye | Orekhovo-Borisovo Yuzhnoye | Aeroport |
| district_Ostankinsky | Ostankinsky | Aeroport |
| district_Otradnoye | Otradnoye | Aeroport |
| district_Perovo | Perovo | Aeroport |
| district_Pervomayskoye | Pervomayskoye | Aeroport |
| district_Pokrovskoye-Streshnevo | Pokrovskoye-Streshnevo | Aeroport |
| district_Preobrazhenskoye | Preobrazhenskoye | Aeroport |
| district_Presnensky | Presnensky | Aeroport |
| district_Prospekt Vernadskogo | Prospekt Vernadskogo | Aeroport |
| district_Ramenki | Ramenki | Aeroport |
| district_Rogovskoye | Rogovskoye | Aeroport |
| district_Rostokino | Rostokino | Aeroport |
| district_Ryazanovskoye | Ryazanovskoye | Aeroport |
| district_Savyolovsky | Savyolovsky | Aeroport |
| district_Severnoye Butovo | Severnoye Butovo | Aeroport |
| district_Severnoye Izmaylovo | Severnoye Izmaylovo | Aeroport |
| district_Severnoye Medvedkovo | Severnoye Medvedkovo | Aeroport |
| district_Severnoye Tushino | Severnoye Tushino | Aeroport |
| district_Severny | Severny | Aeroport |
| district_Shchapovskoye | Shchapovskoye | Aeroport |
| district_Shcherbinka | Shcherbinka | Aeroport |
| district_Shchukino | Shchukino | Aeroport |
| district_Sokol | Sokol | Aeroport |
| district_Sokolinaya Gora | Sokolinaya Gora | Aeroport |
| district_Sokolniki | Sokolniki | Aeroport |
| district_Solntsevo | Solntsevo | Aeroport |
| district_Sosenskoye | Sosenskoye | Aeroport |
| district_Strogino | Strogino | Aeroport |
| district_Sviblovo | Sviblovo | Aeroport |
| district_Tagansky | Tagansky | Aeroport |
| district_Timiryazevsky | Timiryazevsky | Aeroport |
| district_Troitsk | Troitsk | Aeroport |
| district_Troparyovo-Nikulino | Troparyovo-Nikulino | Aeroport |
| district_Tsaritsyno | Tsaritsyno | Aeroport |
| district_Tverskoy | Tverskoy | Aeroport |
| district_Tyoply Stan | Tyoply Stan | Aeroport |
| district_Veshnyaki | Veshnyaki | Aeroport |
| district_Vnukovo | Vnukovo | Aeroport |
| district_Vnukovskoye | Vnukovskoye | Aeroport |
| district_Voronovskoye | Voronovskoye | Aeroport |
| district_Voskresenskoye | Voskresenskoye | Aeroport |
| district_Vostochnoye Izmaylovo | Vostochnoye Izmaylovo | Aeroport |
| district_Vostochny | Vostochny | Aeroport |
| district_Voykovsky | Voykovsky | Aeroport |
| district_Yakimanka | Yakimanka | Aeroport |
| district_Yaroslavsky | Yaroslavsky | Aeroport |
| district_Yasenevo | Yasenevo | Aeroport |
| district_Yuzhnoye Butovo | Yuzhnoye Butovo | Aeroport |
| district_Yuzhnoye Medvedkovo | Yuzhnoye Medvedkovo | Aeroport |
| district_Yuzhnoye Tushino | Yuzhnoye Tushino | Aeroport |
| district_Zamoskvorechye | Zamoskvorechye | Aeroport |
| district_Zapadnoye Degunino | Zapadnoye Degunino | Aeroport |
| district_Zelenograd | Zelenograd | Aeroport |
| district_Zyablikovo | Zyablikovo | Aeroport |
| district_Zyuzino | Zyuzino | Aeroport |

## 9. Verificación final de train y test

Se comprueban alineación, tipos numéricos, finitud, ausencia de nulos y exclusión del objetivo y metadatos. Un ensayo con categorías nominales artificialmente desconocidas verifica que no se agreguen columnas; no altera test. Las referencias temporales e IDs permanecen en `meta_train` y `meta_test`, alineadas por índice.

In [9]:
assert X_train.columns.equals(X_test.columns)
assert X_train.index.equals(y_train.index) and X_test.index.equals(y_test.index)
assert X_train.index.equals(meta_train.index) and X_test.index.equals(meta_test.index)
assert X_train.columns.is_unique
for X in [X_train, X_test]:
    assert all(is_numeric_dtype(t) for t in X.dtypes)
    assert not X.isna().any().any() and np.isfinite(X.to_numpy()).all()
    assert not {"id", "price_rub", "date_posted_dt", *columnas_eliminadas}.intersection(X.columns)
pd.testing.assert_series_equal(y_train, train_crudo.price_rub)
pd.testing.assert_series_equal(y_test, test_crudo.price_rub)
# Ensayo del contrato para categorías desconocidas: solo se usa una copia artificial de train.
sonda = train_codificacion.iloc[:1].copy()
for col in nominales: sonda[col] = "__CATEGORIA_NUEVA_DE_PRUEBA__"
with warnings.catch_warnings():
    warnings.filterwarnings("ignore", message="Found unknown categories", category=UserWarning)
    sonda_codificada = codificador.transform(sonda)
assert sonda_codificada.shape[1] == len(columnas_generadas) and np.isfinite(sonda_codificada).all()
dimensiones_finales = pd.DataFrame([{"Objeto": n, "Filas": len(d), "Columnas": d.shape[1] if d.ndim == 2 else 1}
    for n, d in [("X_train", X_train), ("X_test", X_test), ("y_train", y_train), ("y_test", y_test)]])

**Dimensiones finales para modelado**

| Objeto | Filas | Columnas |
| --- | --- | --- |
| X_train | 40.000 | 163 |
| X_test | 10.000 | 163 |
| y_train | 40.000 | 1 |
| y_test | 10.000 | 1 |

## 10. Exportación de datasets procesados

Se guardan `train_ols_usadas.csv` y `test_ols_usadas.csv` con el mismo orden: predictores numéricos y, al final, `price_rub` original. No se guarda el índice ni se sobrescribe el CSV integrado. Los archivos se leen nuevamente y se verifican dimensiones, columnas, tipos y valores.

Se conserva un único Joblib con los transformadores y sus parámetros, junto con `metadatos_train_ols_usadas.csv` y `metadatos_test_ols_usadas.csv`. Estos últimos mantienen ID y fecha en el mismo orden de filas que cada dataset de modelado. Se recarga el artefacto y se comprueba que reproduce `X_test` sin llamar a `fit`.

In [10]:
carpeta_salida.mkdir(parents=True, exist_ok=True)
train_exportado = X_train.assign(price_rub=y_train)
test_exportado = X_test.assign(price_rub=y_test)
columnas_exportadas = [*columnas_finales, "price_rub"]
assert train_exportado.columns.tolist() == test_exportado.columns.tolist() == columnas_exportadas
def guardar_dataset_necesario(datos, ruta, **opciones):
    if not ruta.exists():
        datos.to_csv(ruta, index=False, **opciones)
        return
    existente = pd.read_csv(ruta)
    assert existente.shape == datos.shape and existente.columns.tolist() == datos.columns.tolist()
    if "date_posted_dt" in datos:
        existente["date_posted_dt"] = pd.to_datetime(existente.date_posted_dt)
        pd.testing.assert_frame_equal(existente, datos.reset_index(drop=True))
    else:
        np.testing.assert_allclose(existente.to_numpy(), datos.to_numpy(), rtol=1e-12, atol=1e-12)
        np.testing.assert_array_equal(existente.price_rub, datos.price_rub.to_numpy())

guardar_dataset_necesario(train_exportado, carpeta_salida / "train_ols_usadas.csv")
guardar_dataset_necesario(test_exportado, carpeta_salida / "test_ols_usadas.csv")
guardar_dataset_necesario(meta_train, carpeta_salida / "metadatos_train_ols_usadas.csv", date_format="%Y-%m-%d")
guardar_dataset_necesario(meta_test, carpeta_salida / "metadatos_test_ols_usadas.csv", date_format="%Y-%m-%d")
registro_transformaciones["columnas_exportadas"] = columnas_exportadas
registro_transformaciones["tipos_predictores"] = X_train.dtypes.astype(str).to_dict()
claves_recuperacion = ['version_flujo', 'entrada', 'sha256_entrada', 'versiones', 'columnas_originales', 'tipos_originales', 'columnas_numericas_entrada', 'correcciones', 'imputacion', 'eliminadas', 'alias', 'nominales', 'referencias', 'binarias', 'ordinales', 'desconocidas', 'dummies', 'temporales', 'entrada_codificador', 'columnas_generadas', 'constantes_excluidas', 'columnas_finales', 'objetivo', 'metadatos', 'columnas_exportadas', 'tipos_predictores']
registro_recuperable = {k: a_json(registro_transformaciones[k]) for k in claves_recuperacion}
ruta_transformadores = carpeta_salida / "transformadores_ols_usadas.joblib"
if not ruta_transformadores.exists():
    joblib.dump({"registro": registro_recuperable, "codificador": codificador}, ruta_transformadores, compress=3)

train_releido = pd.read_csv(carpeta_salida / "train_ols_usadas.csv")
test_releido = pd.read_csv(carpeta_salida / "test_ols_usadas.csv")
assert train_releido.columns.equals(test_releido.columns)
assert train_releido.dtypes.equals(test_releido.dtypes)
for releido, original in [(train_releido, train_exportado), (test_releido, test_exportado)]:
    assert releido.shape == original.shape
    assert releido.columns.tolist() == columnas_exportadas
    assert all(is_numeric_dtype(t) for t in releido.dtypes)
    assert not releido.isna().any().any() and np.isfinite(releido.to_numpy()).all()
    np.testing.assert_allclose(releido.to_numpy(), original.to_numpy(), rtol=1e-12, atol=1e-12)
    np.testing.assert_array_equal(releido.price_rub.to_numpy(), original.price_rub.to_numpy())
artefacto_recargado = joblib.load(ruta_transformadores)
registro_recargado = artefacto_recargado["registro"]
assert registro_recargado == registro_recuperable
assert registro_recargado == a_json(artefacto_recargado["registro"])
def transformar_desde_registro(datos_crudos, parametros, codificador_ajustado):
    # No llama a fit ni calcula estadísticos sobre los nuevos datos.
    datos = datos_crudos.copy(deep=True)
    datos["date_posted_dt"] = pd.to_datetime(datos["date_posted"], errors="coerce")
    if datos.date_posted_dt.isna().any():
        raise ValueError("Fechas inválidas al aplicar parámetros guardados.")
    datos, _ = limpiar_validar(datos, parametros["columnas_numericas_entrada"])
    datos = aplicar_imputacion(datos, {c: p["valor"] for c, p in parametros["imputacion"].items()})
    comprobar_consistencia(datos)
    datos = datos.drop(columns=parametros["eliminadas"], errors="ignore").rename(columns=parametros["alias"])
    objetivo = datos["price_rub"].copy()
    metadatos = datos[parametros["metadatos"]].copy()
    caracteristicas = generar_temporales(datos)
    caracteristicas, _ = aplicar_binarias(caracteristicas, parametros["binarias"])
    caracteristicas = caracteristicas[parametros["entrada_codificador"]]
    with warnings.catch_warnings():
        warnings.filterwarnings("ignore", message="Found unknown categories", category=UserWarning)
        transformada = codificador_ajustado.transform(caracteristicas)
    X = pd.DataFrame(transformada, index=datos.index, columns=parametros["columnas_generadas"])
    return X[parametros["columnas_finales"]], objetivo, metadatos

test_reproducido, y_reproducido, meta_reproducido = transformar_desde_registro(
    test_crudo, registro_recargado, artefacto_recargado["codificador"])
pd.testing.assert_frame_equal(test_reproducido, X_test)
pd.testing.assert_series_equal(y_reproducido, y_test)
pd.testing.assert_frame_equal(meta_reproducido, meta_test)
for nombre, meta in [("train", meta_train), ("test", meta_test)]:
    recuperado = pd.read_csv(carpeta_salida / f"metadatos_{nombre}_ols_usadas.csv")
    assert recuperado.id.tolist() == meta.id.tolist()
    assert pd.to_datetime(recuperado.date_posted_dt).tolist() == meta.date_posted_dt.tolist()
assert hashlib.sha256(entrada.read_bytes()).hexdigest() == registro_recargado["sha256_entrada"]
tabla_exportacion = pd.DataFrame([{"Archivo": f"{nombre}_ols_usadas.csv", "Filas": len(datos),
    "Columnas": datos.shape[1], "Tipos": "Todas numéricas; objetivo sin transformar", "Nulos": int(datos.isna().sum().sum())}
    for nombre, datos in [("train", train_releido), ("test", test_releido)]])

**Archivos de modelado verificados tras su lectura**

| Archivo | Filas | Columnas | Tipos | Nulos |
| --- | --- | --- | --- | --- |
| train_ols_usadas.csv | 40.000 | 164 | Todas numéricas; objetivo sin transformar | 0 |
| test_ols_usadas.csv | 10.000 | 164 | Todas numéricas; objetivo sin transformar | 0 |